# ImageNet Training Loop

In this notebook, we will run an ImageNet training loop on a single GPU in AWS. Eventually we will create a .py file to train ImageNet on multiple GPUs in AWS or RunPod.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../..'))
import time

In [ ]:
import torch
import torchvision
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from utils.imagenet import get_train_transform, get_val_transform
from utils.metrics import accuracy, topk_accuracy

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device, torch.cuda.device_count())

# Hyperparameters

In [ ]:
# Mount network volume
bucket_path = '/workspace/imagenet'

# path to save model
save_path = os.getcwd()

# single epoch for testing. Eventually will set to 90
num_epochs = 4

# may need to be smaller if OOM occurs
batch_size = 2 # 128

# Change depending on number of CPUs, optimize
num_workers = 4
pin_memory = True

# How many batches before logging loss
log_every = 2 # 1000

# How many epochs before checkpointing model
checkpoint_every = 10

# How many times to perform validation per epoch
val_per_epoch = 2

# Optimizer hyperparameters
opt_kwargs = {'lr': 0.1 * batch_size/256, 'momentum': 0.9}
num_warmup = 5
T_max = 90 - num_warmup
eta_min = 1e-5

# Fixed for ImageNet Dataset
C = 3
H, W = 224, 224
num_classes = 1000

# Create model

In [ ]:
model = torchvision.models.resnet50().to(device)

# Import dataset

In [ ]:
#train_ds = torchvision.datasets.ImageFolder(bucket_path + "/train", transform=get_train_transform())
#val_ds = torchvision.datasets.ImageFolder(bucket_path + "/val", transform=get_val_transform())

In [ ]:
t1 = time.time()
val_ds = torchvision.datasets.ImageNet(bucket_path, split='val', transform=get_val_transform())
t2 = time.time()
print(f"Time to create validation dataset: {t2-t1}")

In [ ]:
t1 = time.time()
train_ds = torchvision.datasets.ImageNet(bucket_path, split='train', transform=get_train_transform())
t2 = time.time()
print(f"Time to create training dataset: {t2-t1}")

# Create dataloader

In [ ]:
train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, 
    num_workers=num_workers, pin_memory=pin_memory, drop_last=True
)
print(f"Length of training dataloader is {len(train_dl)}")

In [ ]:
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=2*batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, drop_last=False
)
print(f"Length of validation dataloader is {len(val_dl)}")

# Introduce loss and training metrics

In [ ]:
loss_fn = torch.nn.functional.cross_entropy
metrics = [loss_fn, accuracy, topk_accuracy]
train_metrics = [[]] * len(metrics)
val_metrics = [[]] * len(metrics)
running_train_metrics = [torch.tensor(0.0, device=device) for _ in metrics]
running_val_metrics = [torch.tensor(0.0, device=device) for _ in metrics]

# Create optimizer

In [ ]:
opt = SGD(model.parameters(), **opt_kwargs)

# Create learning rate scheduler

In [ ]:
scheduler1 = LinearLR(opt, 0.01, 1.0, num_warmup)
scheduler2 = CosineAnnealingLR(opt, T_max=T_max, eta_min=eta_min)
scheduler = SequentialLR(opt, schedulers=[scheduler1, scheduler2], milestones=[num_warmup])

# Create loss scaler for mixed-precision training

In [ ]:
scaler = torch.amp.GradScaler(device.type)

# Function to checkpoint model

In [ ]:
def save_checkpoint(model, optimizer, epoch, train_metrics, val_metrics, path=save_path):
    checkpoint = {
        'epochs': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_metrics': torch.tensor(train_metrics),
        'val_metrics': torch.tensor(val_metrics)
    }
    torch.save(checkpoint, path + '/imagenet-checkpoint.pt')
    print(f"Checkpoint saved after {epoch+1} epochs")

# Write functions for training loop

In [ ]:
############# delete this later
train_dl = [None] * 2
val_dl = [None] * 1
val_ds = [None] * len(val_dl) * batch_size

In [ ]:
def is_log_step(i):
    return i % log_every == 0 and i > 0

def is_validate_step(i):
    return (i * val_per_epoch) % len(train_dl) < val_per_epoch

def is_checkpoint_epoch(epoch):
    return epoch % checkpoint_every == 0 and epoch > 0

In [ ]:
def compute_metrics(metrics, running_metrics, logits, y, reduction='mean'):
    for metric, running_metric in zip(metrics, running_metrics):
        running_metric += metric(logits, y, reduction=reduction)

def log_metrics(records, metrics, div=1):
    log(records, metrics, div=div)
    reset_metric(metrics)

def log(records, metrics, div=1):
    for record, metric in zip(records, metrics):
        record.append(metric.item() / div)

def reset_metric(metrics):
    for tensor in metrics:
        tensor.fill_(0.0)

In [ ]:
def train_step(model, X, y, opt, scaler):
    with torch.autocast(device.type):
        logits = model(X)
        
    loss = loss_fn(logits, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return logits

def val_step(model, X):
    with torch.autocast(device.type):
        logits = model(X)
    return logits

# Profile model

In [ ]:
def fake_dataloader_to_profile():
    running_metrics = [torch.tensor(0.0, device=device) for _ in metrics]
    for i in range(len(train_dl)):
        X = torch.randn(batch_size, C, H, W, device=device)
        y = torch.randint(low=0, high=num_classes, size=(batch_size,), device=device)  
        if is_log_step(i):
            print("printing something")
        logits = train_step(model, X, y, opt, scaler)
        with torch.no_grad():
            compute_metrics(metrics, running_metrics, logits, y)

def training_epoch_to_profile():
    running_metrics = [torch.tensor(0.0, device=device) for _ in metrics]

    for i, (X, y) in enumerate(train_dl):
        X = X.to(device, non_blocking=pin_memory)
        y = y.to(device, non_blocking=pin_memory)

        if is_log_step(i):
            print("printing something")
    
        logits = train_step(model, X, y, opt, scaler)

        with torch.no_grad():
            compute_metrics(metrics, running_metrics, logits, y)

In [ ]:
def trace_handler(prof):
    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))

In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function, schedule

activities = [ProfilerActivity.CPU] # ,ProfilerActivity.GPU]

with torch.profiler.profile(
    activities=activities,
    schedule=schedule(wait=0, warmup=1, active=2, skip_first=0, repeat=0),
    on_trace_ready=trace_handler
) as prof:
    for epoch in range(3):
        fake_dataloader_to_profile()
        #profile_training_epoch()
        prof.step()

# Run training loop

In [ ]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    #for i, (X, y) in enumerate(train_dl):
    #   X = X.to(device, non_blocking=pin_memory)
    #   y = y.to(device, non_blocking=pin_memory)
    ################## delete this later
    for i in range(len(train_dl)):
        X = torch.randn(batch_size, C, H, W, device=device)
        y = torch.randint(low=0, high=num_classes, size=(batch_size,), device=device)  
        ###################

        if is_log_step(i):
            log_metrics(train_metrics, running_train_metrics, div=log_every)
            print(f"Training loss, accuracy at iteration {i}/{len(train_dl)}: \
                  {train_metrics[0][-1]:.3f}, {train_metrics[1][-1]:.2f}")
        
        logits = train_step(model, X, y, opt, scaler)

        with torch.no_grad():
            compute_metrics(metrics, running_train_metrics, logits, y)

        if is_validate_step(i):
            with torch.no_grad():
                #for i, (X, y) in enumerate(val_dl):
                #   X = X.to(device, non_blocking=pin_memory)
                #   y = y.to(device, non_blocking=pin_memory)

                ######### DELETE THIS LATER
                for j in range(len(val_dl)):
                    X = torch.randn(2*batch_size, C, H, W, device=device)
                    y = torch.randint(low=0, high=num_classes, size=(2*batch_size,), device=device)  
                    ##############

                    logits = val_step(model, X)
                    compute_metrics(metrics, running_val_metrics, logits, y, reduction='sum')
            log_metrics(val_metrics, running_val_metrics, div=len(val_ds))

    scheduler.step()

    if is_checkpoint_epoch(epoch):
        save_checkpoint(model, opt, epoch, train_metrics, val_metrics)

In [ ]:
# finally, save the model and metrics
save_checkpoint(model, opt, epoch, train_metrics, val_metrics)